# SIFLOW · run_0 · Smoke test

Verifies the install, the full unit-test suite (incl. the SUBS check that is the riskiest MDLM integration point), and an end-to-end MDLM load + one-step generation. Run this first; if it passes, the long runs are safe.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** nothing (entry point)

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

In [ ]:
# Full unit-test suite (runs the torch tests on the Colab GPU)
!python -m pytest tests/ -q

In [ ]:
# End-to-end: load MDLM, verify SUBS gives ~0 mass on the mask token, 1-step generate
import torch
from siflow.teacher import MDLMTeacher
from siflow.head import VelocityHead
from siflow.student import Student
from transformers import AutoTokenizer

teacher = MDLMTeacher(dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained("gpt2")
ids = torch.full((2, 32), teacher.mask_index, device=teacher.device)
z, h = teacher.logits_and_hidden(ids)
p_mask = torch.softmax(z.float(), -1)[..., teacher.mask_index].max().item()
print("max prob on mask token (should be ~0):", p_mask)

head = VelocityHead(teacher.hidden_dim, teacher.embedding_matrix, bottleneck=1024).to(teacher.device)
student = Student(teacher, head)
out = student.generate(2, 32, k=1)
print(tok.decode(out[0].tolist()))
print("smoke OK")

In [ ]:
import json, os
os.makedirs("results", exist_ok=True)
json.dump({"smoke": "ok", "mask_prob": float(p_mask)}, open("results/smoke_ok.json", "w"))

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---
!cp -r results {BASE}/ 2>/dev/null || true
print('saved to', BASE)